# TMEW-1 / HMWMC v2 — Colab and Jupyter runner

Runs the repaired v2 curriculum (`tmew1_run_v2.py`) at scale: multi-seed full runs with the
zero-shot emergent-capability probes, aggregated across seeds, plus the Phase-1 branch
experiments (`tmew1_branch_runner_v2.py`).

**Before running on Colab:** *Runtime → Change runtime type → GPU* (T4 or better). A full
4-epoch curriculum takes about 25 min on an RTX 3070 and roughly 45–60 min on a T4; the
checkpoint for each tier is about 150 MB.

Outputs for every run go to `OUT_ROOT/<run_name>/`:

- `val_tier{N}_ep{E}.json` — validation metrics per tier and epoch;
- `diagnostics_tier{N}.json` — handoff-bucket recall, belief revision, and the emergent probes;
- `checkpoint_tier{N}.pt`.

Mount Google Drive in section 2 so these survive the Colab session.

## 1. Clone repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/ai-wes/hmwmc.git"
REPO_DIR = "hmwmc"
BRANCH = "codex/research-release-readme-ci"   # carries the 2026-11-03 fixes; switch to "main" once merged

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR}/ already exists — pulling latest...")
    !cd {REPO_DIR} && git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q --branch {BRANCH} {REPO_URL}

os.chdir(REPO_DIR)
!git log --oneline -1
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Colab ships PyTorch with CUDA; this only fills in the small extras.
!pip install -q -r requirements.txt

## 2. Verify CUDA and choose where outputs go

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    print(f"Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU.")

In [ ]:
# Mount Drive on Colab so runs persist; anywhere else, fall back to a local folder.
import os

USE_DRIVE = True
OUT_ROOT = "runs"
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        OUT_ROOT = "/content/drive/MyDrive/hmwmc_runs"
    except ImportError:
        print("Not on Colab — writing runs to ./runs")
os.makedirs(OUT_ROOT, exist_ok=True)
print("OUT_ROOT =", OUT_ROOT)

## 3. Smoke test (about 30 s)

Checks execution only; its metrics are not results. The CI check also fails on any `nan`/`inf` or skipped-backward signal.

In [ ]:
!python tmew1_run_v2.py --smoke --out-dir {OUT_ROOT}/smoke 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -40

## 4. Multi-seed full curriculum with emergent probes

One sequential run per seed (the model needs ~3 GB of GPU memory, so parallel seeds on a T4 are not worth it).
Edit `SEEDS`, `EPOCHS`, `TRAIN_EPISODES`, and `BATCH_SIZE` to scale up. Each run prints the per-tier validation
snapshot and the diagnostics block, including the probes:

- HPM slot specialization (>0.3 interesting)
- causal thread separation (>0.1)
- retroactive cause binding (>0.05)
- state reactivation / replay (>0)
- event boundary chunking and surprise–event correlation (>1.5)

In [ ]:
import subprocess, sys, os, time

SEEDS = [0, 1, 2]
EPOCHS = 4
TRAIN_EPISODES = 2048
BATCH_SIZE = 32
RUN_PREFIX = "v2"          # run directories become OUT_ROOT/v2_seed{seed}

for seed in SEEDS:
    out_dir = os.path.join(OUT_ROOT, f"{RUN_PREFIX}_seed{seed}")
    if os.path.exists(os.path.join(out_dir, "diagnostics_tier3.json")):
        print(f"seed {seed}: already complete at {out_dir}, skipping")
        continue
    os.makedirs(out_dir, exist_ok=True)
    cmd = [sys.executable, "-u", "tmew1_run_v2.py",
           "--seed", str(seed), "--epochs", str(EPOCHS),
           "--train-episodes", str(TRAIN_EPISODES), "--batch-size", str(BATCH_SIZE),
           "--out-dir", out_dir]
    print(f"\n=== seed {seed} → {out_dir} ===\n  {' '.join(cmd)}")
    t0 = time.time()
    with open(os.path.join(out_dir, "run.log"), "w") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            if ("UserWarning" in line) or ("return inner" in line) or line.startswith("sys:1"):
                continue
            if ("[val]" in line) or ("qacc/" in line) or ("Diagnostics" in line) or line.startswith("  ") or ("promot" in line) or ("Non-finite" in line):
                print(line, end="")
        rc = proc.wait()
    print(f"seed {seed} finished rc={rc} in {(time.time() - t0) / 60:.1f} min")

## 5. Aggregate across seeds

Mean and standard deviation (ddof=1) over completed seeds for the final-epoch Tier 3 validation metrics, the holder-query handoff buckets, and the emergent probes.

In [ ]:
!python tmew1_aggregate_runs.py "{OUT_ROOT}/{RUN_PREFIX}_seed*" --tier 3 --json {OUT_ROOT}/{RUN_PREFIX}_summary_tier3.json

## 6. Larger configurations

`tmew1_run_v2.py` builds the default model (`d_model=256`, 6 layers, 32 memory slots, 64 episodic slots).
For harder worlds, longer horizons, or HPM variants use the branch runner below, which composes
`WorldConfig`/tier/HPM overrides on top of the same `run_curriculum`. Example: longer Tier 3 horizon and more epochs.

In [ ]:
# Example: Tier 3 at T=96 with 6 epochs, seed 0. Add --seed N to repeat across seeds.
!python tmew1_branch_runner_v2.py --branch baseline \
    --tier3-episode-length 96 --epochs 6 --seed 0 \
    --out-dir {OUT_ROOT}/baseline_T96_seed0 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -60

## 7. Phase-1 branch experiments

Each branch changes one axis (A: harder worlds, B: harder queries, C: HPM architecture, D/E: memory and
stability variants) and is judged against a frozen baseline record by its promotion rubric
(`tmew1_experiments_v2.py`). Run the baseline first; it writes `val.json`, which the other branches read.
Branches run sequentially here because of GPU memory.

Exit codes: 0 promote · 1 kill · 2 undecided · 3 missing capabilities · 4 crash.

In [ ]:
# Step 1: frozen baseline reference (full curriculum from scratch; add --resume <checkpoint_tier2.pt> to shortcut)
BASELINE_DIR = os.path.join(OUT_ROOT, "experiments", "baseline")
!python tmew1_branch_runner_v2.py --branch baseline --out-dir {BASELINE_DIR} 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -40

In [ ]:
# Step 2: branches, sequentially
import subprocess, sys, os
from tmew1_experiments_v2 import BRANCH_IDS

BRANCHES = ["A1", "A2", "C1", "C2"]      # any of BRANCH_IDS; B-branches need --ack-missing-capabilities
EXTRA_ARGS = []                          # e.g. ["--seed", "1"]
BASELINE_RECORD = os.path.join(BASELINE_DIR, "val.json")
assert os.path.exists(BASELINE_RECORD), "run the baseline cell first"

for branch in BRANCHES:
    assert branch in BRANCH_IDS, branch
    out_dir = os.path.join(OUT_ROOT, "experiments", branch)
    os.makedirs(out_dir, exist_ok=True)
    cmd = [sys.executable, "tmew1_branch_runner_v2.py", "--branch", branch,
           "--baseline-record", BASELINE_RECORD, "--out-dir", out_dir, *EXTRA_ARGS]
    if branch.startswith("B"):
        cmd.append("--ack-missing-capabilities")
    print(f"\n=== {branch} ===")
    with open(os.path.join(out_dir, "run.log"), "w") as log:
        rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
    status = {0: "promote", 1: "kill", 2: "undecided", 3: "missing capabilities", 4: "crash"}.get(rc, f"code={rc}")
    print(f"{branch} → {status}   (log: {out_dir}/run.log)")

In [ ]:
# Step 3: inspect verdicts
import json, glob, os

print(f"{'Branch':<12} {'Verdict':<12} {'Target metric':<34} {'Value':>8}")
print("─" * 70)
for d in sorted(glob.glob(os.path.join(OUT_ROOT, "experiments", "*/"))):
    branch = os.path.basename(d.rstrip("/"))
    verdict_files = sorted(glob.glob(os.path.join(d, "verdict_*.jsonl")))
    if not verdict_files:
        print(f"{branch:<12} {'no verdict':<12}")
        continue
    with open(verdict_files[-1]) as f:
        lines = [json.loads(l) for l in f if l.strip()]
    rec = lines[-1] if lines else {}
    v = rec.get("verdict", "?")
    target = (rec.get("config") or {}).get("rubric", {}).get("target_metric", "—")
    val = (rec.get("val_record") or {}).get(target, float("nan")) if target != "—" else float("nan")
    print(f"{branch:<12} {v:<12} {target:<34} {val:>8.4f}")